In [1]:
from pathlib import Path

import geopandas as gpd
import rasterio
import numpy as np
import pandas as pd
import os

from rasterio.mask import mask
from rasterio.transform import xy
from tqdm import tqdm

In [2]:
# Rutas del proyecto

directorio_actual = os.getcwd()
directorio_padre = os.path.dirname(directorio_actual)

BASE_DIR = Path(directorio_padre)

RASTER_DIR = BASE_DIR / "raster"
LABELS_DIR = BASE_DIR / "labels"
METADATA_DIR = BASE_DIR / "metadata"
TABLES_DIR = BASE_DIR / "tables"

for folder in [RASTER_DIR, LABELS_DIR, METADATA_DIR, TABLES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)


raster_path = RASTER_DIR / "olaroz_sentinel2_l2a_seca_2025_mosaico_rgb_natural_rectangular.tif"
scl_path = RASTER_DIR / "olaroz_sentinel2_l2a_seca_2025_scl_control_rectangular.tif"
polygons_path = LABELS_DIR / "olaroz_training_polygons.geojson"

In [3]:
# Bandas del TIFF multibanda, en el orden en que fue exportado
band_names = [
    "B04", "B03", "B02",
    "B05", "B06", "B07",
    "B08", "B8A", "B11", "B12"
]

print("Raster principal:", raster_path)
print("Raster SCL:", scl_path)
print("Polígonos:", polygons_path)

Raster principal: C:\Users\pablonicolasr\Desktop\pablonicolas\educacion_formal\mentodatos2026\repo\mentodatos_pdis\raster\olaroz_sentinel2_l2a_seca_2025_mosaico_rgb_natural_rectangular.tif
Raster SCL: C:\Users\pablonicolasr\Desktop\pablonicolas\educacion_formal\mentodatos2026\repo\mentodatos_pdis\raster\olaroz_sentinel2_l2a_seca_2025_scl_control_rectangular.tif
Polígonos: C:\Users\pablonicolasr\Desktop\pablonicolas\educacion_formal\mentodatos2026\repo\mentodatos_pdis\labels\olaroz_training_polygons.geojson


In [4]:
# Diccionario de clases


class_dictionary = pd.DataFrame([
    {
        "class_id": 1,
        "class_name": "salina",
        "macro_class": "sal",
        "description": "Superficie salina clara o costra salina homogenea",
    },
    {
        "class_id": 2,
        "class_name": "suelo_desnudo",
        "macro_class": "sin_vegetacion",
        "description": "Suelos desnudos, bordes aluviales, sedimentos, laderas secas o roca expuesta",
    },
    {
        "class_id": 3,
        "class_name": "agua_humedad",
        "macro_class": "humedad_agua",
        "description": "Agua superficial, salmuera, piletas humedas o zonas oscuras saturadas",
    },
    {
        "class_id": 4,
        "class_name": "suelo_rocoso",
        "macro_class": "sin_vegetacion",
        "description": "Superficie rugosa, roca expuesta o laderas rocosas",
    },
])

class_dictionary_path = METADATA_DIR / "class_dictionary.csv"
class_dictionary.to_csv(class_dictionary_path, index=False, encoding="utf-8")

class_dictionary

,class_id,class_name,macro_class,description
0,1,salina,sal,Superficie salina clara o costra salina homogenea
1,2,suelo_desnudo,sin_vegetacion,"Suelos desnudos, bordes aluviales, sedimentos,..."
2,3,agua_humedad,humedad_agua,"Agua superficial, salmuera, piletas humedas o ..."
3,4,suelo_rocoso,sin_vegetacion,"Superficie rugosa, roca expuesta o laderas roc..."


In [5]:
# Leer y validar polígonos GeoJSON

gdf = gpd.read_file(polygons_path)

print("Cantidad de polígonos:", len(gdf))
print("CRS original:", gdf.crs)
print("Columnas:", list(gdf.columns))
print(gdf.head())

Cantidad de polígonos: 70
CRS original: EPSG:4326
Columnas: ['fid', 'sample_id', 'class_id', 'class_name', 'macro_class', 'season', 'confidence', 'split', 'source', 'notes', 'geometry']
   fid         sample_id  class_id    class_name   macro_class season  \
0    1  agua_humedad_001         3  agua_humedad  humedad_agua   seca   
1    2  agua_humedad_002         3  agua_humedad  humedad_agua   seca   
2    3  agua_humedad_003         3  agua_humedad  humedad_agua   seca   
3    4  agua_humedad_004         3  agua_humedad  humedad_agua   seca   
4    5  agua_humedad_005         3  agua_humedad  humedad_agua   seca   

  confidence  split              source  \
0       alta  train  sentinel2_l2a_2025   
1       alta  train  sentinel2_l2a_2025   
2       alta  train  sentinel2_l2a_2025   
3       alta  train  sentinel2_l2a_2025   
4       alta  train  sentinel2_l2a_2025   

                                               notes  \
0  zona oscura homogénea asociada a humedad super...   
1  z

In [6]:
# Validaciones básicas del GeoJSON

required_columns = [
    "sample_id",
    "class_id",
    "class_name",
    "macro_class",
    "season",
    "confidence",
    "split",
    "source",
    "notes",
    "geometry",
]

missing_columns = [col for col in required_columns if col not in gdf.columns]

if missing_columns:
    raise ValueError(f"Faltan columnas en el GeoJSON: {missing_columns}")

# Normalizar textos
for col in ["sample_id", "class_name", "macro_class", "season", "confidence", "split", "source", "notes"]:
    gdf[col] = gdf[col].astype(str).str.strip()

expected_classes = {
    "salina": 1,
    "suelo_desnudo": 2,
    "agua_humedad": 3,
    "suelo_rocoso": 4,
}

expected_macro = {
    "salina": "sal",
    "suelo_desnudo": "sin_vegetacion",
    "agua_humedad": "humedad_agua",
    "suelo_rocoso": "sin_vegetacion",
}

expected_splits = {"train", "validation", "test"}

# Validar clases
unexpected_classes = sorted(set(gdf["class_name"]) - set(expected_classes.keys()))

if unexpected_classes:
    raise ValueError(f"Hay clases no esperadas: {unexpected_classes}")

# Validar split
unexpected_splits = sorted(set(gdf["split"]) - expected_splits)

if unexpected_splits:
    raise ValueError(f"Hay valores no esperados en split: {unexpected_splits}")

# Reforzar class_id y macro_class según class_name
gdf["class_id"] = gdf["class_name"].map(expected_classes).astype(int)
gdf["macro_class"] = gdf["class_name"].map(expected_macro)

# Validar geometrías
invalid_count = (~gdf.is_valid).sum()
print("Geometrías inválidas:", invalid_count)

if invalid_count > 0:
    print("Corrigiendo geometrías inválidas con make_valid()...")
    gdf["geometry"] = gdf.geometry.make_valid()

# Reportes
print("\nPolígonos por clase:")
print(gdf["class_name"].value_counts())

print("\nPolígonos por split:")
print(gdf["split"].value_counts())

print("\nPolígonos por clase y split:")
print(gdf.groupby(["class_name", "split"]).size())

Geometrías inválidas: 0

Polígonos por clase:
class_name
agua_humedad     20
salina           20
suelo_desnudo    20
suelo_rocoso     10
Name: count, dtype: int64

Polígonos por split:
split
train         49
test          11
validation    10
Name: count, dtype: int64

Polígonos por clase y split:
class_name     split     
agua_humedad   test           3
               train         14
               validation     3
salina         test           3
               train         14
               validation     3
suelo_desnudo  test           3
               train         14
               validation     3
suelo_rocoso   test           2
               train          7
               validation     1
dtype: int64


In [7]:
# Reproyectar polígonos al CRS del raster


with rasterio.open(raster_path) as src:
    raster_crs = src.crs
    raster_count = src.count
    raster_nodata = src.nodata
    raster_shape = (src.height, src.width)
    raster_bounds = src.bounds

print("CRS raster:", raster_crs)
print("Cantidad de bandas:", raster_count)
print("NoData raster:", raster_nodata)
print("Shape raster:", raster_shape)
print("Bounds raster:", raster_bounds)

if raster_count != len(band_names):
    raise ValueError(
        f"El raster tiene {raster_count} bandas, pero band_names tiene {len(band_names)} nombres."
    )

gdf = gdf.to_crs(raster_crs)

print("CRS polígonos reproyectados:", gdf.crs)

CRS raster: EPSG:32719
Cantidad de bandas: 10
NoData raster: nan
Shape raster: (2412, 1879)
Bounds raster: BoundingBox(left=718210.0, bottom=7378970.0, right=755790.0, top=7427210.0)
CRS polígonos reproyectados: PROJCS["WGS 84 / UTM zone 19S",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",-69],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",10000000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","32719"]]


In [10]:
# Extraer píxeles del TIFF dentro de cada polígono

rows = []

with rasterio.open(raster_path) as src, rasterio.open(scl_path) as scl_src:

    # Validar CRS y dimensiones entre raster espectral y SCL
    if src.crs != scl_src.crs:
        raise ValueError(f"CRS diferentes: raster={src.crs}, scl={scl_src.crs}")

    if src.res != scl_src.res:
        print("Advertencia: resolución diferente entre raster y SCL.")
        print("Raster res:", src.res)
        print("SCL res:", scl_src.res)

    for idx, poly in tqdm(gdf.iterrows(), total=len(gdf), desc="Extrayendo píxeles"):

        geom = [poly.geometry]

        try:
            data, out_transform = mask(
                src,
                geom,
                crop=True,
                filled=True
            )

            scl_data, scl_transform = mask(
                scl_src,
                geom,
                crop=True,
                filled=True
            )

        except Exception as e:
            print("Error en polígono:", poly.get("sample_id"), e)
            continue

        # data: bandas, alto, ancho
        data = data.astype("float32")
        scl = scl_data[0]

        # Máscara inicial de píxeles válidos
        valid = np.ones(data.shape[1:], dtype=bool)

        # Descartar píxeles con NaN o infinitos
        valid &= np.all(np.isfinite(data), axis=0)

        # Descartar píxeles todo cero, por seguridad
        valid &= ~(np.all(data == 0, axis=0))

        if valid.sum() == 0:
            print("Polígono sin píxeles válidos:", poly.get("sample_id"))
            continue

        pixels = data[:, valid].T
        scl_values = scl[valid]

        # Coordenadas x, y de cada píxel válido
        valid_rows, valid_cols = np.where(valid)
        xs, ys = xy(out_transform, valid_rows, valid_cols, offset="center")

        for pixel, scl_value, x_coord, y_coord in zip(pixels, scl_values, xs, ys):

            record = {
                "sample_id": poly["sample_id"],
                "class_id": int(poly["class_id"]),
                "class_name": poly["class_name"],
                "macro_class": poly["macro_class"],
                "season": poly["season"],
                "confidence": poly["confidence"],
                "split": poly["split"],
                "source": poly["source"],
                "notes": poly["notes"],
                "scl": int(scl_value),
                "x": float(x_coord),
                "y": float(y_coord),
            }

            for band_name, value in zip(band_names, pixel):
                record[band_name] = float(value)

            rows.append(record)

df_raw = pd.DataFrame(rows)

print("Dataset crudo:")
print(df_raw.head())
print(df_raw.shape)

print("\nPíxeles por clase:")
print(df_raw["class_name"].value_counts())

print("\nPíxeles por clase y split:")
print(df_raw.groupby(["class_name", "split"]).size())

raw_csv = TABLES_DIR / "olaroz_samples_pixels_raw.csv"
raw_parquet = TABLES_DIR / "olaroz_samples_pixels_raw.parquet"

df_raw.to_csv(raw_csv, index=False, encoding="utf-8")
print("CSV guardado:", raw_csv)

print("Guardado:", raw_csv)
print("Guardado:", raw_parquet)

Extrayendo píxeles: 100%|█████████████████████████████████████████████████████████████| 70/70 [00:00<00:00, 141.24it/s]


Dataset crudo:
          sample_id  class_id    class_name   macro_class season confidence  \
0  agua_humedad_001         3  agua_humedad  humedad_agua   seca       alta   
1  agua_humedad_001         3  agua_humedad  humedad_agua   seca       alta   
2  agua_humedad_001         3  agua_humedad  humedad_agua   seca       alta   
3  agua_humedad_001         3  agua_humedad  humedad_agua   seca       alta   
4  agua_humedad_001         3  agua_humedad  humedad_agua   seca       alta   

   split              source  \
0  train  sentinel2_l2a_2025   
1  train  sentinel2_l2a_2025   
2  train  sentinel2_l2a_2025   
3  train  sentinel2_l2a_2025   
4  train  sentinel2_l2a_2025   

                                               notes  scl  ...     B04  \
0  zona oscura homogénea asociada a humedad super...    5  ...  2964.0   
1  zona oscura homogénea asociada a humedad super...    5  ...  2814.0   
2  zona oscura homogénea asociada a humedad super...    5  ...  2873.0   
3  zona oscura homogé

In [12]:
# Filtrar píxeles malos usando SCL


df = pd.read_csv(TABLES_DIR / "olaroz_samples_pixels_raw.csv")

# Valores problemáticos de SCL
# 0  = NoData
# 1  = Saturated/defective
# 3  = Cloud shadows
# 7  = Unclassified
# 8  = Cloud medium probability
# 9  = Cloud high probability
# 10 = Thin cirrus
# 11 = Snow/ice
#
# Se dejan pasar:
# 2 = Dark area pixels
# 4 = Vegetation
# 5 = Not vegetated / bare soil
# 6 = Water
#
# En tu caso dejamos 2 porque agua_humedad puede contener zonas oscuras/húmedas.
bad_scl = [0, 1, 3, 7, 8, 9, 10, 11]

df_filtered = df[~df["scl"].isin(bad_scl)].copy()

print("Antes de filtrar:", df.shape)
print("Después de filtrar:", df_filtered.shape)

print("\nDistribución SCL después del filtro:")
print(df_filtered["scl"].value_counts().sort_index())

print("\nPíxeles por clase después del filtro:")
print(df_filtered["class_name"].value_counts())

print("\nPíxeles por clase y split después del filtro:")
print(df_filtered.groupby(["class_name", "split"]).size())

pixels_csv = TABLES_DIR / "olaroz_samples_pixels.csv"
pixels_parquet = TABLES_DIR / "olaroz_samples_pixels.parquet"

df_filtered.to_csv(pixels_csv, index=False, encoding="utf-8")
print("CSV guardado:", pixels_csv)

print("Guardado:", pixels_csv)
print("Guardado:", pixels_parquet)

Antes de filtrar: (21507, 22)
Después de filtrar: (13878, 22)

Distribución SCL después del filtro:
scl
2       5
5    7328
6    6545
Name: count, dtype: int64

Píxeles por clase después del filtro:
class_name
agua_humedad     6717
suelo_rocoso     4432
suelo_desnudo    2729
Name: count, dtype: int64

Píxeles por clase y split después del filtro:
class_name     split     
agua_humedad   test           140
               train         6076
               validation     501
suelo_desnudo  test           194
               train         2346
               validation     189
suelo_rocoso   test           584
               train         3356
               validation     492
dtype: int64
CSV guardado: C:\Users\pablonicolasr\Desktop\pablonicolas\educacion_formal\mentodatos2026\repo\mentodatos_pdis\tables\olaroz_samples_pixels.csv
Guardado: C:\Users\pablonicolasr\Desktop\pablonicolas\educacion_formal\mentodatos2026\repo\mentodatos_pdis\tables\olaroz_samples_pixels.csv
Guardado: C:\Users\pab

In [13]:
# Calcular índices espectrales


df = pd.read_csv(TABLES_DIR / "olaroz_samples_pixels.csv")

eps = 1e-6

# Índices clásicos
df["NDVI"] = (df["B08"] - df["B04"]) / (df["B08"] + df["B04"] + eps)
df["NDWI"] = (df["B03"] - df["B08"]) / (df["B03"] + df["B08"] + eps)
df["MNDWI"] = (df["B03"] - df["B11"]) / (df["B03"] + df["B11"] + eps)

# Índices/ratios útiles para suelo, humedad y salares
df["NDBI_like"] = (df["B11"] - df["B08"]) / (df["B11"] + df["B08"] + eps)
df["SWIR_ratio"] = df["B11"] / (df["B12"] + eps)
df["NIR_SWIR_ratio"] = df["B08"] / (df["B11"] + eps)

# Ratios adicionales útiles para separar brillo/salinidad/suelo
df["RED_SWIR_ratio"] = df["B04"] / (df["B11"] + eps)
df["GREEN_SWIR_ratio"] = df["B03"] / (df["B11"] + eps)
df["BLUE_RED_ratio"] = df["B02"] / (df["B04"] + eps)

# Limpiar infinitos y NaN
df = df.replace([np.inf, -np.inf], np.nan).dropna().copy()

features_csv = TABLES_DIR / "olaroz_samples_features.csv"
features_parquet = TABLES_DIR / "olaroz_samples_features.parquet"

df.to_csv(features_csv, index=False, encoding="utf-8")
print("CSV guardado:", features_csv)

print("Dataset con índices:")
print(df.head())
print(df.shape)

print("\nPíxeles por clase:")
print(df["class_name"].value_counts())

print("\nPíxeles por clase y split:")
print(df.groupby(["class_name", "split"]).size())

print("Guardado:", features_csv)
print("Guardado:", features_parquet)

CSV guardado: C:\Users\pablonicolasr\Desktop\pablonicolas\educacion_formal\mentodatos2026\repo\mentodatos_pdis\tables\olaroz_samples_features.csv
Dataset con índices:
          sample_id  class_id    class_name   macro_class season confidence  \
0  agua_humedad_001         3  agua_humedad  humedad_agua   seca       alta   
1  agua_humedad_001         3  agua_humedad  humedad_agua   seca       alta   
2  agua_humedad_001         3  agua_humedad  humedad_agua   seca       alta   
3  agua_humedad_001         3  agua_humedad  humedad_agua   seca       alta   
4  agua_humedad_001         3  agua_humedad  humedad_agua   seca       alta   

   split              source  \
0  train  sentinel2_l2a_2025   
1  train  sentinel2_l2a_2025   
2  train  sentinel2_l2a_2025   
3  train  sentinel2_l2a_2025   
4  train  sentinel2_l2a_2025   

                                               notes  scl  ...     B12  \
0  zona oscura homogénea asociada a humedad super...    5  ...  1096.5   
1  zona oscura ho

In [14]:
# Crear versión limpia


df = pd.read_csv(TABLES_DIR / "olaroz_samples_features.csv")

# Usar solo muestras de alta confianza
df_clean = df[df["confidence"].str.lower() == "alta"].copy()

# Revalidar clases
expected_class_names = ["salina", "suelo_desnudo", "agua_humedad", "suelo_rocoso"]
df_clean = df_clean[df_clean["class_name"].isin(expected_class_names)].copy()

# Revalidar splits
df_clean = df_clean[df_clean["split"].isin(["train", "validation", "test"])].copy()

# Eliminar duplicados exactos si existieran
df_clean = df_clean.drop_duplicates().copy()

clean_csv = TABLES_DIR / "olaroz_samples_clean.csv"
clean_parquet = TABLES_DIR / "olaroz_samples_clean.parquet"

df_clean.to_csv(clean_csv, index=False, encoding="utf-8")
print("CSV guardado:", df_clean)

print("Dataset limpio:")
print(df_clean.shape)

print("\nPíxeles por clase:")
print(df_clean["class_name"].value_counts())

print("\nPíxeles por clase y split:")
print(df_clean.groupby(["class_name", "split"]).size())

print("Guardado:", clean_csv)
print("Guardado:", clean_parquet)

CSV guardado:               sample_id  class_id    class_name     macro_class season  \
0      agua_humedad_001         3  agua_humedad    humedad_agua   seca   
1      agua_humedad_001         3  agua_humedad    humedad_agua   seca   
2      agua_humedad_001         3  agua_humedad    humedad_agua   seca   
3      agua_humedad_001         3  agua_humedad    humedad_agua   seca   
4      agua_humedad_001         3  agua_humedad    humedad_agua   seca   
...                 ...       ...           ...             ...    ...   
13873  suelo_rocoso_010         4  suelo_rocoso  sin_vegetacion   seca   
13874  suelo_rocoso_010         4  suelo_rocoso  sin_vegetacion   seca   
13875  suelo_rocoso_010         4  suelo_rocoso  sin_vegetacion   seca   
13876  suelo_rocoso_010         4  suelo_rocoso  sin_vegetacion   seca   
13877  suelo_rocoso_010         4  suelo_rocoso  sin_vegetacion   seca   

      confidence  split              source  \
0           alta  train  sentinel2_l2a_2025   
1  

In [15]:
# Crear versión balanceada respetando split

df = pd.read_csv(TABLES_DIR / "olaroz_samples_clean.csv")

balanced_parts = []

for split_name, split_df in df.groupby("split"):

    counts = split_df["class_name"].value_counts()

    print(f"\nSplit: {split_name}")
    print(counts)

    min_n = counts.min()

    split_balanced = (
        split_df.groupby("class_name", group_keys=False)
        .apply(lambda x: x.sample(min_n, random_state=42))
        .reset_index(drop=True)
    )

    balanced_parts.append(split_balanced)

df_balanced = pd.concat(balanced_parts, ignore_index=True)

balanced_csv = TABLES_DIR / "olaroz_samples_balanced.csv"
balanced_parquet = TABLES_DIR / "olaroz_samples_balanced.parquet"

df_balanced.to_csv(balanced_csv, index=False, encoding="utf-8")
print("CSV guardado:", df_balanced)

print("\nDataset balanceado:")
print(df_balanced.shape)

print("\nDistribución balanceada por clase:")
print(df_balanced["class_name"].value_counts())

print("\nDistribución balanceada por clase y split:")
print(df_balanced.groupby(["class_name", "split"]).size())

print("Guardado:", balanced_csv)
print("Guardado:", balanced_parquet)


Split: test
class_name
suelo_rocoso     584
suelo_desnudo    194
agua_humedad     140
Name: count, dtype: int64

Split: train
class_name
agua_humedad     6076
suelo_rocoso     3356
suelo_desnudo    2346
Name: count, dtype: int64

Split: validation
class_name
agua_humedad     501
suelo_rocoso     492
suelo_desnudo    189
Name: count, dtype: int64
CSV guardado:              sample_id  class_id    class_name     macro_class season  \
0     agua_humedad_019         3  agua_humedad    humedad_agua   seca   
1     agua_humedad_019         3  agua_humedad    humedad_agua   seca   
2     agua_humedad_019         3  agua_humedad    humedad_agua   seca   
3     agua_humedad_020         3  agua_humedad    humedad_agua   seca   
4     agua_humedad_019         3  agua_humedad    humedad_agua   seca   
...                ...       ...           ...             ...    ...   
8020  suelo_rocoso_008         4  suelo_rocoso  sin_vegetacion   seca   
8021  suelo_rocoso_008         4  suelo_rocoso  sin_v

In [16]:
# Guardar reporte resumen del dataset

df_features = pd.read_csv(TABLES_DIR / "olaroz_samples_features.csv")
df_clean = pd.read_csv(TABLES_DIR / "olaroz_samples_clean.csv")
df_balanced = pd.read_csv(TABLES_DIR / "olaroz_samples_balanced.csv")

report_path = METADATA_DIR / "dataset_report.txt"

with open(report_path, "w", encoding="utf-8") as f:
    f.write("REPORTE DEL DATASET - SALAR DE OLAROZ\n")
    f.write("=====================================\n\n")

    f.write("Clases utilizadas:\n")
    f.write("1 - salina\n")
    f.write("2 - suelo_desnudo\n")
    f.write("3 - agua_humedad\n")
    f.write("4 - suelo_rocoso\n\n")

    f.write("Bandas utilizadas:\n")
    f.write(", ".join(band_names) + "\n\n")

    f.write("Índices calculados:\n")
    indices = [
        "NDVI", "NDWI", "MNDWI", "NDBI_like",
        "SWIR_ratio", "NIR_SWIR_ratio",
        "RED_SWIR_ratio", "GREEN_SWIR_ratio", "BLUE_RED_ratio"
    ]
    f.write(", ".join(indices) + "\n\n")

    f.write("Dataset features:\n")
    f.write(str(df_features.shape) + "\n")
    f.write(str(df_features["class_name"].value_counts()) + "\n\n")

    f.write("Dataset limpio:\n")
    f.write(str(df_clean.shape) + "\n")
    f.write(str(df_clean["class_name"].value_counts()) + "\n\n")

    f.write("Dataset balanceado:\n")
    f.write(str(df_balanced.shape) + "\n")
    f.write(str(df_balanced["class_name"].value_counts()) + "\n\n")

    f.write("Distribución por clase y split - dataset balanceado:\n")
    f.write(str(df_balanced.groupby(["class_name", "split"]).size()) + "\n")

print("Reporte guardado en:", report_path)

Reporte guardado en: C:\Users\pablonicolasr\Desktop\pablonicolas\educacion_formal\mentodatos2026\repo\mentodatos_pdis\metadata\dataset_report.txt


In [17]:
# Vista final

df = pd.read_csv(TABLES_DIR / "olaroz_samples_balanced.csv")

feature_columns = [
    "B04", "B03", "B02",
    "B05", "B06", "B07",
    "B08", "B8A", "B11", "B12",
    "NDVI", "NDWI", "MNDWI",
    "NDBI_like", "SWIR_ratio", "NIR_SWIR_ratio",
    "RED_SWIR_ratio", "GREEN_SWIR_ratio", "BLUE_RED_ratio"
]

target_column = "class_id"

print("Columnas predictoras:")
print(feature_columns)

print("\nVariable objetivo:")
print(target_column)

print("\nShape final:")
print(df.shape)

print("\nClases:")
print(df[["class_id", "class_name", "macro_class"]].drop_duplicates().sort_values("class_id"))

print("\nDistribución final:")
print(df.groupby(["class_name", "split"]).size())

Columnas predictoras:
['B04', 'B03', 'B02', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B11', 'B12', 'NDVI', 'NDWI', 'MNDWI', 'NDBI_like', 'SWIR_ratio', 'NIR_SWIR_ratio', 'RED_SWIR_ratio', 'GREEN_SWIR_ratio', 'BLUE_RED_ratio']

Variable objetivo:
class_id

Shape final:
(8025, 31)

Clases:
     class_id     class_name     macro_class
140         2  suelo_desnudo  sin_vegetacion
0           3   agua_humedad    humedad_agua
280         4   suelo_rocoso  sin_vegetacion

Distribución final:
class_name     split     
agua_humedad   test           140
               train         2346
               validation     189
suelo_desnudo  test           140
               train         2346
               validation     189
suelo_rocoso   test           140
               train         2346
               validation     189
dtype: int64
